# Silver PolicyData Implementation

Reads unmerged bronze policies, cleans and validates them, MERGEs valid rows into silver,
quarantines bad rows, and flags processed bronze rows.

| Outcome | Destination | Bronze `merge_flag` |
|---|---|---|
| Valid | `silver.policydata` | `true` |
| Hard reject (bad data) | `silver.quarantine_policydata` | `true` |
| Soft reject: customer not (yet) in `silver.customer` | stays in bronze | `false` (retried next run) |

**Job order:** depends on Silver_Customer. Silver_Claim depends on it.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS policysystemdatabricks.silver.policydata (
  policy_id         INT,
  policy_type       STRING,
  customer_id       INT,
  start_date        TIMESTAMP,
  end_date          TIMESTAMP,
  premium           DOUBLE,
  coverage_amount   DOUBLE,
  merged_timestamp  TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@insurancepolicy1adls.dfs.core.windows.net/policydata';

CREATE TABLE IF NOT EXISTS policysystemdatabricks.silver.quarantine_policydata (
  policy_id             INT,
  policy_type           STRING,
  customer_id           INT,
  start_date            TIMESTAMP,
  end_date              TIMESTAMP,
  premium               DOUBLE,
  coverage_amount       DOUBLE,
  reject_reason         STRING,
  quarantined_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@insurancepolicy1adls.dfs.core.windows.net/quarantine/policydata';

In [0]:
from pyspark.sql import functions as F, Window

COLUMNS = ["policy_id", "policy_type", "customer_id", "start_date", "end_date", "premium", "coverage_amount"]
SOFT_REASONS = ["missing_customer"]  # parent may arrive later -> retry instead of quarantine

# 1. Read only unmerged bronze policies
policy_df = spark.sql("SELECT * FROM policysystemdatabricks.bronze.policydata WHERE merge_flag = false")

# 2. Clean: trim / collapse spaces and InitCap policy_type, default missing policy_type
clean_df = policy_df.selectExpr(
    "policy_id",
    r"coalesce(nullif(initcap(regexp_replace(trim(policy_type), '\\s+', ' ')), ''), 'Unknown') AS policy_type",
    "customer_id",
    "start_date",
    "end_date",
    "premium",
    "coverage_amount",
)

# 3. Deduplicate: one row per policy_id (latest start_date). Extra sort keys make the pick deterministic.
#    NULL policy_id rows are kept individually so each one is quarantined.
w = Window.partitionBy("policy_id").orderBy(
    F.col("start_date").desc_nulls_last(), F.col("end_date").desc_nulls_last(),
    F.col("premium").desc_nulls_last(), F.col("coverage_amount").desc_nulls_last())
df_result = (clean_df.withColumn("rn", F.row_number().over(w))
             .filter("rn = 1 OR policy_id IS NULL")
             .drop("rn"))

In [0]:
# Referential check against SILVER customers: silver must only point to silver.
silver_customers = (spark.table("policysystemdatabricks.silver.customer")
                    .select("customer_id").distinct()
                    .withColumn("_customer_exists", F.lit(True)))

reject_reason = (
    F.when(F.col("policy_id").isNull(), "null_policy_id")
     .when(F.col("customer_id").isNull(), "null_customer_id")
     .when(F.col("start_date").isNull() | F.col("end_date").isNull(), "missing_dates")
     .when(F.col("end_date") <= F.col("start_date"), "end_not_after_start")
     .when(F.col("premium").isNull() | (F.col("premium") <= 0), "invalid_premium")
     .when(F.col("coverage_amount").isNull() | (F.col("coverage_amount") <= 0), "invalid_coverage_amount")
     .when(F.col("_customer_exists").isNull(), "missing_customer")
)

classified_df = (df_result
                 .join(silver_customers, on="customer_id", how="left")
                 .withColumn("reject_reason", reject_reason)
                 .select(*COLUMNS, "reject_reason"))
classified_df.createOrReplaceTempView("policy_classified")

display(classified_df.groupBy(F.coalesce("reject_reason", F.lit("VALID")).alias("outcome")).count())

outcome,count
invalid_coverage_amount,45
missing_customer,448
null_customer_id,51
missing_dates,107


In [0]:
display(spark.sql("""
MERGE INTO policysystemdatabricks.silver.policydata AS t
USING (SELECT * FROM policy_classified WHERE reject_reason IS NULL) AS s
ON t.policy_id = s.policy_id
WHEN MATCHED THEN UPDATE SET
  t.policy_type      = s.policy_type,
  t.customer_id      = s.customer_id,
  t.start_date       = s.start_date,
  t.end_date         = s.end_date,
  t.premium          = s.premium,
  t.coverage_amount  = s.coverage_amount,
  t.merged_timestamp = current_timestamp()
WHEN NOT MATCHED THEN INSERT
  (policy_id, policy_type, customer_id, start_date, end_date, premium, coverage_amount, merged_timestamp)
  VALUES (s.policy_id, s.policy_type, s.customer_id, s.start_date, s.end_date, s.premium, s.coverage_amount, current_timestamp())
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
soft_list = ", ".join(f"'{r}'" for r in SOFT_REASONS)

display(spark.sql(f"""
INSERT INTO policysystemdatabricks.silver.quarantine_policydata
SELECT {", ".join(COLUMNS)}, reject_reason, current_timestamp()
FROM policy_classified
WHERE reject_reason IS NOT NULL AND reject_reason NOT IN ({soft_list})
"""))

num_affected_rows,num_inserted_rows
203,203


In [0]:
display(spark.sql(f"""
MERGE INTO policysystemdatabricks.bronze.policydata AS b
USING (SELECT DISTINCT policy_id FROM policy_classified
       WHERE policy_id IS NOT NULL
         AND (reject_reason IS NULL OR reject_reason NOT IN ({soft_list}))) AS s
ON b.policy_id = s.policy_id AND b.merge_flag = false
WHEN MATCHED THEN UPDATE SET b.merge_flag = true
"""))

# NULL keys cannot be matched in a MERGE; they are always quarantined above, so flag them directly.
display(spark.sql("""
UPDATE policysystemdatabricks.bronze.policydata
SET merge_flag = true
WHERE merge_flag = false AND policy_id IS NULL
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
609,609,0,0


num_affected_rows
0


In [0]:
%sql
SELECT (SELECT count(*) FROM policysystemdatabricks.bronze.policydata WHERE merge_flag = false) AS bronze_waiting_for_customer,
       (SELECT count(*) FROM policysystemdatabricks.silver.quarantine_policydata) AS quarantined_rows_total,
       count(*)                                  AS silver_rows,
       count(DISTINCT policy_id)                 AS silver_distinct_policies,
       count_if(policy_type = 'Unknown')         AS default_policy_types,
       count_if(merged_timestamp IS NULL)        AS rows_missing_merged_timestamp,  -- should be 0
       (SELECT count(*) FROM policysystemdatabricks.silver.policydata p
        LEFT ANTI JOIN policysystemdatabricks.silver.customer c ON p.customer_id = c.customer_id) AS orphan_customers,  -- should be 0
       count_if(NOT (end_date > start_date) OR start_date IS NULL OR end_date IS NULL) AS date_outliers,  -- should be 0
       count_if(NOT (premium > 0 AND coverage_amount > 0) OR premium IS NULL OR coverage_amount IS NULL) AS bad_amounts  -- should be 0
FROM policysystemdatabricks.silver.policydata

bronze_waiting_for_customer,quarantined_rows_total,silver_rows,silver_distinct_policies,default_policy_types,rows_missing_merged_timestamp,orphan_customers,date_outliers,bad_amounts
1344,203,349,349,10,0,0,0,0
